In [1]:
import os
import glob
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad

# Assuming MLAB environment variable is set for the base path
PATH = os.path.join(os.getenv("MLAB"), "projects/brcameta/projects/sig_recon")

# Merging Prediction Results

In [3]:
splits_df = pd.read_csv(os.path.join(PATH, "data/sigs/perturb-seq/pb_splits.csv"))

# Making control adata first

In [5]:
rpe1_adata = sc.read(os.path.join(PATH, "data/scgpt/rpe1_shared/perturb_processed.h5ad"))
k562_adata = sc.read(os.path.join(PATH, "data/scgpt/k562_shared/perturb_processed.h5ad"))

In [15]:
rpe1_ctrl = rpe1_adata[rpe1_adata.obs.gene == "non-targeting"].copy()
k562_ctrl = k562_adata[k562_adata.obs.gene == "non-targeting"].copy()

In [16]:
# Change adata.var
k562_ctrl.var['ensembl_id'] = k562_ctrl.var.index.copy()
# Then set gene names as index, if there are duplicates we append a numeric
k562_ctrl.var.index = k562_ctrl.var['gene_name'].astype(str)
k562_ctrl.var_names_make_unique()
k562_ctrl.var.drop('gene_name', axis=1, inplace=True)

rpe1_ctrl.var['ensembl_id'] = rpe1_ctrl.var.index.copy()
# Then set gene names as index, if there are duplicates we append a numeric
rpe1_ctrl.var.index = rpe1_ctrl.var['gene_name'].astype(str)
rpe1_ctrl.var_names_make_unique()
rpe1_ctrl.var.drop('gene_name', axis=1, inplace=True)

In [17]:
# Changing for raw too
if k562_ctrl.raw is not None:
    # Get the raw data
    raw_adata = k562_ctrl.raw.to_adata()
    raw_adata.var['ensembl_id'] = raw_adata.var.index.copy()
    raw_adata.var_names = raw_adata.var['gene_name'].astype(str)
    raw_adata.var_names_make_unique()
    raw_adata.var.drop('gene_name', axis=1, inplace=True)
    
    k562_ctrl.raw = raw_adata

# For rpe1
if rpe1_ctrl.raw is not None:
    raw_adata = rpe1_ctrl.raw.to_adata()
    raw_adata.var['ensembl_id'] = raw_adata.var.index.copy()
    raw_adata.var_names = raw_adata.var['gene_name'].astype(str)
    raw_adata.var_names_make_unique()
    raw_adata.var.drop('gene_name', axis=1, inplace=True)
    
    rpe1_ctrl.raw = raw_adata

## 10th

In [4]:
k562_adata_paths = glob.glob(os.path.join(PATH, "data/scgpt/prediction_results/k562_10th/*.h5ad"))
rpe1_adata_paths = glob.glob(os.path.join(PATH, "data/scgpt/prediction_results/rpe1_10th/*.h5ad"))

In [18]:
k562_adatas = [sc.read_h5ad(path) for path in k562_adata_paths]
rpe1_adatas = [sc.read_h5ad(path) for path in rpe1_adata_paths]

/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: 

In [19]:
is_duplicate = k562_adatas[1].var_names.duplicated(keep=False)

# Get the positions and names
dup_df = pd.DataFrame({
    'gene': k562_adatas[1].var_names[is_duplicate],
    'original_index': np.where(is_duplicate)[0]
})
print(dup_df)

     gene  original_index
0  HSPA14            2113
1  HSPA14            2475


In [20]:
# Fix duplicates in the lists of adatas
for adata in k562_adatas:
    adata.var_names_make_unique()

for adata in rpe1_adatas:
    adata.var_names_make_unique()

In [21]:
k562_unique_perts = [set(adata.obs.perturbation.unique()) for adata in k562_adatas]
rpe1_unique_perts = [set(adata.obs.perturbation.unique()) for adata in rpe1_adatas]

In [22]:
k562_dup_counts = [adata.obs["perturbation"].duplicated().sum() for adata in k562_adatas]
rpe1_dup_counts = [adata.obs["perturbation"].duplicated().sum() for adata in rpe1_adatas]

print(f"K562 duplicate counts: {k562_dup_counts}")
print(f"RPE1 duplicate counts: {rpe1_dup_counts}")

K562 duplicate counts: [np.int64(1327), np.int64(1355), np.int64(1335), np.int64(1303), np.int64(1344), np.int64(1382), np.int64(1314), np.int64(1366), np.int64(1335), np.int64(1334)]
RPE1 duplicate counts: [np.int64(2441), np.int64(2441), np.int64(2404), np.int64(2443), np.int64(2467), np.int64(2402), np.int64(2428), np.int64(2427), np.int64(2390), np.int64(2444)]


In [23]:
from functools import reduce
k562_unique_perts_intersection = reduce(lambda a, b: a & b, k562_unique_perts)
rpe1_unique_perts_intersection = reduce(lambda a, b: a & b, rpe1_unique_perts)

In [24]:
# There should be no overlapping perturbations because of the splits file
assert(k562_unique_perts_intersection == set())
assert(rpe1_unique_perts_intersection == set())

In [25]:
# 1. Concatenate the k562 list
k562_adatas_merged = ad.concat(k562_adatas, join="inner")
k562_adatas_merged.obs_names = k562_adatas_merged.obs["perturbation"]
# 2. Concatenate the rpe1 list
rpe1_adatas_merged = ad.concat(rpe1_adatas, join="inner")
rpe1_adatas_merged.obs_names = rpe1_adatas_merged.obs["perturbation"]

/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:812: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    ['EIF2S1+ctrl', 'ZNF720+ctrl', 'CTU2+ctrl', 'MFN2+ctrl', 'ERCC2+ctrl']

    Inferred to be: categorical

  names = self._prep_dim_index(names, "obs")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:812: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    ['DHX37+ctrl', 'HSPA5+ctrl', 'PSMD6+ctrl', '

In [26]:
assert(len(set(k562_adatas_merged.var_names) & set(k562_ctrl.var_names)) == len(set(k562_adatas_merged.var_names)))
assert(len(set(rpe1_adatas_merged.var_names) & set(rpe1_ctrl.var_names)) == len(set(rpe1_adatas_merged.var_names)))

In [27]:
# 3. Merge K562 merged with RPE1 control
# (Using inner join to ensure gene sets match between the two cell lines)
k562_with_rpe1_ctrl = ad.concat(
    {"perturbed_k562": k562_adatas_merged, "ctrl_rpe1": rpe1_ctrl},
    label="group",
    join="inner"
)

# 4. Merge RPE1 merged with K562 control
rpe1_with_k562_ctrl = ad.concat(
    {"perturbed_rpe1": rpe1_adatas_merged, "ctrl_k562": k562_ctrl},
    label="group",
    join="inner"
)

/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/merge.py:1434: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/merge.py:1434: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [28]:
k562_with_rpe1_ctrl.write(os.path.join(PATH, "data/scgpt/prediction_results/k562_rpe1_with_rpe1_ctrl_10th.h5ad"))
rpe1_with_k562_ctrl.write(os.path.join(PATH, "data/scgpt/prediction_results/rpe1_k562_with_k562_ctrl_10th.h5ad"))

## 90th

In [33]:
k562_adata_paths = glob.glob(os.path.join(PATH, "data/scgpt/prediction_results/k562_90th/*.h5ad"))
rpe1_adata_paths = glob.glob(os.path.join(PATH, "data/scgpt/prediction_results/rpe1_90th/*.h5ad"))

In [34]:
k562_adatas = [sc.read_h5ad(path) for path in k562_adata_paths]
rpe1_adatas = [sc.read_h5ad(path) for path in rpe1_adata_paths]

/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: 

In [35]:
is_duplicate = k562_adatas[1].var_names.duplicated(keep=False)

# Get the positions and names
dup_df = pd.DataFrame({
    'gene': k562_adatas[1].var_names[is_duplicate],
    'original_index': np.where(is_duplicate)[0]
})
print(dup_df)

     gene  original_index
0  HSPA14            2113
1  HSPA14            2475


In [36]:
# Fix duplicates in the lists of adatas
for adata in k562_adatas:
    adata.var_names_make_unique()

for adata in rpe1_adatas:
    adata.var_names_make_unique()

In [37]:
k562_unique_perts = [set(adata.obs.perturbation.unique()) for adata in k562_adatas]
rpe1_unique_perts = [set(adata.obs.perturbation.unique()) for adata in rpe1_adatas]

In [38]:
k562_dup_counts = [adata.obs["perturbation"].duplicated().sum() for adata in k562_adatas]
rpe1_dup_counts = [adata.obs["perturbation"].duplicated().sum() for adata in rpe1_adatas]

print(f"K562 duplicate counts: {k562_dup_counts}")
print(f"RPE1 duplicate counts: {rpe1_dup_counts}")

K562 duplicate counts: [np.int64(192), np.int64(144), np.int64(154), np.int64(147), np.int64(133), np.int64(148), np.int64(147), np.int64(149), np.int64(124), np.int64(135)]
RPE1 duplicate counts: [np.int64(273), np.int64(266), np.int64(254), np.int64(253), np.int64(253), np.int64(265), np.int64(257), np.int64(265), np.int64(288), np.int64(290)]


In [39]:
from functools import reduce
k562_unique_perts_intersection = reduce(lambda a, b: a & b, k562_unique_perts)
rpe1_unique_perts_intersection = reduce(lambda a, b: a & b, rpe1_unique_perts)

In [40]:
# There should be no overlapping perturbations because of the splits file
assert(k562_unique_perts_intersection == set())
assert(rpe1_unique_perts_intersection == set())

In [41]:
# 1. Concatenate the k562 list
k562_adatas_merged = ad.concat(k562_adatas, join="inner")
k562_adatas_merged.obs_names = k562_adatas_merged.obs["perturbation"]
# 2. Concatenate the rpe1 list
rpe1_adatas_merged = ad.concat(rpe1_adatas, join="inner")
rpe1_adatas_merged.obs_names = rpe1_adatas_merged.obs["perturbation"]

/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:812: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    ['TFAM+ctrl', 'SOD2+ctrl', 'AP2M1+ctrl', 'SUPV3L1+ctrl', 'CYFIP1+ctrl']

    Inferred to be: categorical

  names = self._prep_dim_index(names, "obs")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:812: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    ['GTF2F2+ctrl', 'RPL11+ctrl', 'PFN1+ctrl', 

In [42]:
assert(len(set(k562_adatas_merged.var_names) & set(k562_ctrl.var_names)) == len(set(k562_adatas_merged.var_names)))
assert(len(set(rpe1_adatas_merged.var_names) & set(rpe1_ctrl.var_names)) == len(set(rpe1_adatas_merged.var_names)))

In [43]:
# 3. Merge K562 merged with RPE1 control
# (Using inner join to ensure gene sets match between the two cell lines)
k562_with_rpe1_ctrl = ad.concat(
    {"perturbed_k562": k562_adatas_merged, "ctrl_rpe1": rpe1_ctrl},
    label="group",
    join="inner"
)

# 4. Merge RPE1 merged with K562 control
rpe1_with_k562_ctrl = ad.concat(
    {"perturbed_rpe1": rpe1_adatas_merged, "ctrl_k562": k562_ctrl},
    label="group",
    join="inner"
)

/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/merge.py:1434: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/merge.py:1434: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(
/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [44]:
k562_with_rpe1_ctrl.write(os.path.join(PATH, "data/scgpt/prediction_results/k562_rpe1_with_rpe1_ctrl_90th.h5ad"))
rpe1_with_k562_ctrl.write(os.path.join(PATH, "data/scgpt/prediction_results/rpe1_k562_with_k562_ctrl_90th.h5ad"))